# Information Retrieval

## Learning Objectives
1. Implement BM25 scoring from scratch in NumPy and compare it to TF-IDF ranking
2. Train a bi-encoder dense retrieval model with contrastive loss in PyTorch
3. Fuse BM25 and dense rankings with Reciprocal Rank Fusion (RRF) for hybrid retrieval
4. Compute MRR@10, NDCG@10, and Recall@20 from scratch and compare retrieval systems

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from collections import defaultdict
import math
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)
torch.manual_seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

## Level 1: BM25 from Scratch in NumPy

BM25 (Best Match 25) is the gold standard sparse retrieval algorithm used in Elasticsearch,
Solr, and Lucene. It improves on TF-IDF by:
1. **Normalising term frequency**: extra occurrences of a word yield diminishing returns
2. **Document length normalisation**: long documents are penalised to prevent score inflation

The scoring formula for query `q` and document `d`:

```
score(q, d) = sum_t IDF(t) * TF(t,d) * (k1 + 1) / (TF(t,d) + k1 * (1 - b + b * |d| / avgdl))
```

Where `k1=1.2` (TF saturation) and `b=0.75` (length normalisation strength).

In [ ]:
# Level 1: BM25 from scratch

# ── Synthetic corpus ────────────────────────────────────────────────────────
TECH_WORDS  = ['python', 'machine', 'learning', 'neural', 'network', 'model', 'data', 'code']
SPORT_WORDS = ['football', 'team', 'game', 'player', 'score', 'match', 'win', 'goal']
FOOD_WORDS  = ['recipe', 'cook', 'restaurant', 'flavor', 'ingredient', 'taste', 'dish', 'meal']
COMMON_WORDS = ['the', 'a', 'is', 'and', 'in', 'for', 'this', 'that', 'with', 'has']

rng = np.random.default_rng(0)

def make_document(domain_words: list, n_words: int) -> list:
    """Generate a word list with ~60% domain words and ~40% common words."""
    words = []
    for _ in range(n_words):
        if rng.random() < 0.60:
            words.append(rng.choice(domain_words))
        else:
            words.append(rng.choice(COMMON_WORDS))
    return words

# 20 documents: 7 tech, 7 sport, 6 food — varying lengths (5-20 words)
doc_groups = (
    [(TECH_WORDS,  rng.integers(8, 20)) for _ in range(7)] +
    [(SPORT_WORDS, rng.integers(8, 20)) for _ in range(7)] +
    [(FOOD_WORDS,  rng.integers(8, 20)) for _ in range(6)]
)
corpus = [make_document(dw, n) for dw, n in doc_groups]
N_DOCS = len(corpus)

print(f"Corpus: {N_DOCS} documents")
for i, doc in enumerate(corpus):
    print(f"  Doc {i:2d} ({len(doc):2d} words): {' '.join(doc[:8])}...")

# ── BM25 implementation ───────────────────────────────────────────────────
k1, b = 1.2, 0.75

def build_inverted_index(corpus: list) -> dict:
    """Map each term to its document frequencies: {term: {doc_id: count}}."""
    index = defaultdict(lambda: defaultdict(int))
    for doc_id, doc in enumerate(corpus):
        for word in doc:
            index[word][doc_id] += 1
    return index

def bm25_score(query: list, doc_id: int, corpus: list,
               index: dict, k1: float = 1.2, b: float = 0.75) -> float:
    """
    Compute BM25 score for a single (query, document) pair.

    Args:
        query:   list of query tokens
        doc_id:  index into corpus
        corpus:  list of token lists
        index:   inverted index from build_inverted_index
        k1, b:   BM25 hyperparameters

    Returns:
        scalar BM25 score
    """
    N    = len(corpus)
    avgdl = np.mean([len(d) for d in corpus])
    doc_len = len(corpus[doc_id])
    score = 0.0

    for term in query:
        df = len(index.get(term, {}))  # number of docs containing term
        if df == 0:
            continue
        tf = index[term].get(doc_id, 0)
        idf = math.log((N - df + 0.5) / (df + 0.5) + 1.0)
        norm_tf = tf * (k1 + 1) / (tf + k1 * (1 - b + b * doc_len / avgdl))
        score += idf * norm_tf

    return score


def tfidf_score(query: list, doc_id: int, corpus: list, index: dict) -> float:
    """Simple TF-IDF scoring for comparison."""
    N     = len(corpus)
    score = 0.0
    for term in query:
        df = len(index.get(term, {}))
        if df == 0:
            continue
        tf  = index[term].get(doc_id, 0) / (len(corpus[doc_id]) + 1e-8)
        idf = math.log((N + 1) / (df + 1))
        score += tf * idf
    return score


inv_index = build_inverted_index(corpus)

# 5 test queries — one domain-specific, one cross-domain
queries = [
    ['machine', 'learning', 'model'],
    ['football', 'team', 'game'],
    ['recipe', 'cook', 'flavor'],
    ['neural', 'network', 'data'],
    ['python', 'code'],
]

print("\nBM25 vs TF-IDF rankings (top-5 docs per query):")
print("=" * 60)
for q in queries:
    bm25_scores  = [(i, bm25_score(q,  i, corpus, inv_index)) for i in range(N_DOCS)]
    tfidf_scores = [(i, tfidf_score(q, i, corpus, inv_index)) for i in range(N_DOCS)]
    bm25_ranked  = sorted(bm25_scores,  key=lambda x: -x[1])[:5]
    tfidf_ranked = sorted(tfidf_scores, key=lambda x: -x[1])[:5]
    bm25_ids  = [x[0] for x in bm25_ranked]
    tfidf_ids = [x[0] for x in tfidf_ranked]
    print(f"Query: {q}")
    print(f"  BM25  top-5 docs: {bm25_ids}  (scores: {[round(s,2) for _,s in bm25_ranked]})")
    print(f"  TF-IDF top-5 docs: {tfidf_ids}")
    print()

## Level 2: Bi-Encoder Dense Retrieval

Sparse methods (BM25, TF-IDF) match exact keywords. Dense retrieval encodes queries
and documents into embedding vectors and measures **semantic similarity** (cosine).

A **bi-encoder** passes queries and documents through the same encoder independently.
Training uses **contrastive loss** — push positive (query, relevant-doc) pairs together
and negative (query, random-doc) pairs apart in embedding space.

At inference, all document embeddings are pre-computed (indexed), then only the query
needs to be encoded — making it efficient for large-scale retrieval.

In [ ]:
# Level 2: Bi-encoder dense retrieval with contrastive training

VOCAB_SIZE_BI = 200
EMBED_DIM_BI  = 32
HIDDEN_DIM_BI = 64
PAD_ID_BI     = 0

# ── Build vocabulary from corpus ────────────────────────────────────────────
from collections import Counter

all_words_bi = [w for doc in corpus for w in doc]
word_freq = Counter(all_words_bi)
bi_vocab = {w: i+1 for i, (w, _) in enumerate(word_freq.most_common(VOCAB_SIZE_BI-1))}
# Reserve 0 for PAD

def encode_doc(words: list, vocab: dict, max_len: int = 16) -> list:
    """Encode a word list to padded integer ids."""
    ids = [vocab.get(w, 0) for w in words][:max_len]
    ids += [PAD_ID_BI] * max(0, max_len - len(ids))
    return ids

MAX_LEN_BI = 16
corpus_ids = [encode_doc(doc, bi_vocab) for doc in corpus]
corpus_tensor = torch.tensor(corpus_ids, dtype=torch.long)   # [N_DOCS, T]

# ── BiEncoder model ──────────────────────────────────────────────────────────
class BiEncoder(nn.Module):
    """Mean-pooling bi-encoder for dense retrieval."""
    def __init__(self, vocab_size: int = VOCAB_SIZE_BI,
                 embed_dim: int = EMBED_DIM_BI,
                 hidden_dim: int = HIDDEN_DIM_BI):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Embedding(vocab_size, embed_dim),   # will call mean after
        )
        self.proj = nn.Linear(embed_dim, hidden_dim)

    def encode(self, token_ids: torch.Tensor) -> torch.Tensor:
        """
        Encode token ids to fixed-size embeddings.
        Args:
            token_ids: [B, T]
        Returns:
            embeddings: [B, hidden_dim]
        """
        emb = self.encoder[0](token_ids).mean(dim=1)   # mean pooling [B, D]
        return self.proj(emb)                           # [B, H]

    def forward(self, queries: torch.Tensor,
                docs: torch.Tensor) -> torch.Tensor:
        """Compute cosine similarities between query and doc batches."""
        q_emb = self.encode(queries)
        d_emb = self.encode(docs)
        return torch.cosine_similarity(q_emb, d_emb)


# ── Synthetic training pairs ────────────────────────────────────────────────
def make_training_pairs(corpus_ids: list, domain_labels: list,
                         n_pairs: int = 200) -> list:
    """
    Generate (query_ids, pos_doc_ids, neg_doc_ids) triples.
    Positive: query and doc from same domain.
    Negative: query and doc from different domains.
    """
    rng2 = np.random.default_rng(11)
    domain_labels_arr = np.array(domain_labels)
    pairs = []
    for _ in range(n_pairs):
        q_idx = int(rng2.integers(0, len(corpus_ids)))
        q_dom = domain_labels[q_idx]
        same_dom   = np.where(domain_labels_arr == q_dom)[0]
        diff_dom   = np.where(domain_labels_arr != q_dom)[0]
        pos_idx = int(rng2.choice(same_dom))
        neg_idx = int(rng2.choice(diff_dom))
        # Modify query slightly by sampling from same domain words
        q_words = corpus[q_idx].copy()
        pairs.append((corpus_ids[q_idx], corpus_ids[pos_idx], corpus_ids[neg_idx]))
    return pairs

domain_labels = [0]*7 + [1]*7 + [2]*6   # tech=0, sport=1, food=2
train_pairs = make_training_pairs(corpus_ids, domain_labels, n_pairs=200)

bi_model = BiEncoder().to(device)
opt_bi   = optim.Adam(bi_model.parameters(), lr=1e-3)

def contrastive_loss(pos_sim: torch.Tensor, neg_sim: torch.Tensor,
                     margin: float = 0.3) -> torch.Tensor:
    """
    Margin-based contrastive loss:
      L = max(0, neg_sim - pos_sim + margin)
    Pushes pos_sim > neg_sim by at least margin.
    """
    return torch.relu(neg_sim - pos_sim + margin).mean()


BATCH_SIZE_BI = 32
bi_losses = []
for step in range(100):
    idx = np.random.choice(len(train_pairs), BATCH_SIZE_BI, replace=True)
    batch = [train_pairs[i] for i in idx]
    q_ids   = torch.tensor([t[0] for t in batch], dtype=torch.long).to(device)
    pos_ids = torch.tensor([t[1] for t in batch], dtype=torch.long).to(device)
    neg_ids = torch.tensor([t[2] for t in batch], dtype=torch.long).to(device)

    pos_sim = bi_model(q_ids, pos_ids)   # [B]
    neg_sim = bi_model(q_ids, neg_ids)   # [B]
    loss = contrastive_loss(pos_sim, neg_sim)

    opt_bi.zero_grad(); loss.backward(); opt_bi.step()
    bi_losses.append(loss.item())

print(f"Bi-encoder training done. Loss: {bi_losses[0]:.4f} -> {bi_losses[-1]:.4f}")


# ── Dense retrieval: Recall@5 evaluation ────────────────────────────────────
@torch.no_grad()
def dense_retrieve(query_words: list, top_k: int = 5) -> list:
    """Retrieve top-k documents by cosine similarity with bi-encoder."""
    bi_model.eval()
    q_ids   = torch.tensor([encode_doc(query_words, bi_vocab)], dtype=torch.long).to(device)
    doc_ids = corpus_tensor.to(device)                  # [N_DOCS, T]
    q_emb   = bi_model.encode(q_ids)                   # [1, H]
    d_emb   = bi_model.encode(doc_ids)                 # [N_DOCS, H]
    sims    = torch.cosine_similarity(q_emb, d_emb)    # [N_DOCS]
    topk    = sims.argsort(descending=True)[:top_k]
    return topk.cpu().numpy().tolist()

# Recall@5: fraction of relevant docs found in top-5
# Relevant = same domain as query
recall5_scores = []
for q_idx in range(N_DOCS):
    relevant = set(i for i, d in enumerate(domain_labels) if d == domain_labels[q_idx])
    relevant.discard(q_idx)  # exclude query itself
    if not relevant:
        continue
    retrieved = set(dense_retrieve(corpus[q_idx], top_k=5))
    retrieved.discard(q_idx)
    recall5_scores.append(len(retrieved & relevant) / len(relevant))

print(f"Dense retrieval Recall@5 (self-retrieval excluded): {np.mean(recall5_scores):.3f}")

## Real-World Example 1: Hybrid Retrieval with Reciprocal Rank Fusion (RRF)

BM25 excels at keyword queries; dense retrieval handles semantic paraphrases.
**RRF** fuses both ranked lists without requiring score calibration:

```
RRF_score(d) = sum_r 1 / (60 + rank_r(d))
```

The constant 60 controls the influence of highly-ranked documents.
Documents appearing high in both lists get the highest combined scores.

In [ ]:
# Real-World Example 1: Hybrid retrieval with Reciprocal Rank Fusion

def bm25_retrieve(query: list, top_k: int = None) -> list:
    """Return all documents ranked by BM25 score."""
    scores = [(i, bm25_score(query, i, corpus, inv_index)) for i in range(N_DOCS)]
    ranked = sorted(scores, key=lambda x: -x[1])
    if top_k:
        ranked = ranked[:top_k]
    return [doc_id for doc_id, _ in ranked]


def rrf_fusion(rankings: list, k: int = 60) -> list:
    """
    Reciprocal Rank Fusion over multiple ranked lists.

    Args:
        rankings: list of ranked document-id lists (each is a full ranking)
        k:        constant to prevent high impact of very top ranks (default 60)

    Returns:
        doc_ids sorted by RRF score descending
    """
    rrf_scores = defaultdict(float)
    for ranked_list in rankings:
        for rank, doc_id in enumerate(ranked_list):
            rrf_scores[doc_id] += 1.0 / (k + rank + 1)
    return sorted(rrf_scores.keys(), key=lambda d: -rrf_scores[d])


def recall_at_k(query_idx: int, retrieved: list, k: int = 5) -> float:
    """Compute Recall@k given query domain label."""
    relevant = {i for i, d in enumerate(domain_labels)
                if d == domain_labels[query_idx] and i != query_idx}
    if not relevant:
        return 1.0
    return len(set(retrieved[:k]) & relevant) / len(relevant)


# Evaluate all three methods across all queries
bm25_recalls, dense_recalls, hybrid_recalls = [], [], []

for q_idx in range(N_DOCS):
    query_words = corpus[q_idx]

    bm25_ranked  = bm25_retrieve(query_words)
    dense_ranked = dense_retrieve(query_words, top_k=N_DOCS)
    hybrid_ranked = rrf_fusion([bm25_ranked, dense_ranked])

    bm25_recalls.append(recall_at_k(q_idx, bm25_ranked,  k=5))
    dense_recalls.append(recall_at_k(q_idx, dense_ranked, k=5))
    hybrid_recalls.append(recall_at_k(q_idx, hybrid_ranked, k=5))

print("Recall@5 averaged across all queries:")
print(f"  BM25     : {np.mean(bm25_recalls):.3f}")
print(f"  Dense    : {np.mean(dense_recalls):.3f}")
print(f"  Hybrid   : {np.mean(hybrid_recalls):.3f}")
print()
print("RRF explanation:")
print("  A document ranked #1 by BM25 gets 1/(60+1) = 0.0164")
print("  A document ranked #1 by dense also gets 0.0164")
print("  Total if top-ranked by both: 0.0328 — beats any single-ranker document")

# Bar chart
fig, ax = plt.subplots(figsize=(5, 3))
methods = ['BM25', 'Dense', 'Hybrid (RRF)']
vals    = [np.mean(bm25_recalls), np.mean(dense_recalls), np.mean(hybrid_recalls)]
ax.bar(methods, vals, color=['steelblue', 'darkorange', 'seagreen'])
ax.set_ylabel('Recall@5')
ax.set_title('Hybrid vs Single-Method Retrieval')
ax.set_ylim(0, 1)
for i, v in enumerate(vals):
    ax.text(i, v + 0.02, f'{v:.3f}', ha='center', fontsize=10)
plt.tight_layout()
plt.savefig('/tmp/08_rw1_hybrid.png', dpi=100)
plt.close()
print("Plot saved to /tmp/08_rw1_hybrid.png")

## Real-World Example 2: Re-ranking with a Cross-Encoder

A **cross-encoder** sees both query and document tokens together — unlike the
bi-encoder which encodes them independently. This allows full attention across
query-document token pairs, yielding much better relevance judgements.

The two-stage retrieval pipeline:
1. **First stage**: fast BM25 retrieves top-20 candidates
2. **Second stage**: cross-encoder re-ranks the 20 candidates (slower, higher accuracy)

Only the top-20 candidates are re-ranked, keeping overall latency manageable.

In [ ]:
# Real-World Example 2: Cross-encoder re-ranking

class CrossEncoder(nn.Module):
    """
    Cross-encoder: concatenates [query; doc] token ids, then uses a small
    transformer-like MLP to predict relevance score.
    """
    def __init__(self, vocab_size: int = VOCAB_SIZE_BI,
                 embed_dim: int = 32, hidden_dim: int = 64,
                 max_len: int = 32):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, embed_dim, padding_idx=PAD_ID_BI)
        self.pos   = nn.Embedding(max_len, embed_dim)
        enc_layer  = nn.TransformerEncoderLayer(
            embed_dim, nhead=4, dim_feedforward=128, batch_first=True)
        self.transformer = nn.TransformerEncoder(enc_layer, num_layers=1)
        self.head = nn.Linear(embed_dim, 1)   # relevance logit

    def forward(self, q_ids: torch.Tensor,
                d_ids: torch.Tensor) -> torch.Tensor:
        """
        Args:
            q_ids: [B, Tq]  query token ids
            d_ids: [B, Td]  document token ids
        Returns:
            scores: [B] relevance logits
        """
        # Concatenate along sequence dimension
        combined = torch.cat([q_ids, d_ids], dim=1)   # [B, Tq+Td]
        T = combined.size(1)
        pos = torch.arange(T, device=combined.device).unsqueeze(0)
        h   = self.embed(combined) + self.pos(pos)    # [B, T, D]
        h   = self.transformer(h)                      # [B, T, D]
        cls = h[:, 0, :]                               # use first position
        return self.head(cls).squeeze(-1)              # [B]


def make_cross_encoder_data(n_pairs: int = 150) -> tuple:
    """
    Build binary relevance training data: (q_ids, d_ids, label).
    Positive label (1) = same domain, negative label (0) = different domain.
    """
    rng3 = np.random.default_rng(13)
    q_list, d_list, labels = [], [], []
    for _ in range(n_pairs):
        q_idx = int(rng3.integers(0, N_DOCS))
        q_dom = domain_labels[q_idx]
        if rng3.random() > 0.5:
            # Positive
            same = [i for i, d in enumerate(domain_labels) if d == q_dom and i != q_idx]
            if not same:
                continue
            d_idx = int(rng3.choice(same))
            lbl = 1
        else:
            # Negative
            diff = [i for i, d in enumerate(domain_labels) if d != q_dom]
            d_idx = int(rng3.choice(diff))
            lbl = 0
        q_list.append(corpus_ids[q_idx])
        d_list.append(corpus_ids[d_idx])
        labels.append(lbl)
    return (torch.tensor(q_list, dtype=torch.long),
            torch.tensor(d_list, dtype=torch.long),
            torch.tensor(labels, dtype=torch.float32))

MAX_LEN_CE = 16
cross_model = CrossEncoder(max_len=MAX_LEN_CE * 2).to(device)
opt_ce      = optim.Adam(cross_model.parameters(), lr=5e-4)
bce_ce      = nn.BCEWithLogitsLoss()

Xq_ce, Xd_ce, y_ce = make_cross_encoder_data(150)
split_ce = 120
Xq_tr_ce, Xd_tr_ce, y_tr_ce = Xq_ce[:split_ce].to(device), Xd_ce[:split_ce].to(device), y_ce[:split_ce].to(device)
Xq_te_ce, Xd_te_ce, y_te_ce = Xq_ce[split_ce:].to(device), Xd_ce[split_ce:].to(device), y_ce[split_ce:].to(device)

ce_losses = []
for step in range(80):
    cross_model.train()
    scores_tr = cross_model(Xq_tr_ce, Xd_tr_ce)
    loss_ce   = bce_ce(scores_tr, y_tr_ce)
    opt_ce.zero_grad(); loss_ce.backward(); opt_ce.step()
    ce_losses.append(loss_ce.item())

print(f"Cross-encoder training done. Loss: {ce_losses[0]:.4f} -> {ce_losses[-1]:.4f}")


@torch.no_grad()
def rerank_with_cross_encoder(query_words: list, candidate_ids: list,
                               top_k: int = 5) -> list:
    """Re-rank a candidate list using the cross-encoder."""
    cross_model.eval()
    q_ids = torch.tensor([encode_doc(query_words, bi_vocab)] * len(candidate_ids),
                          dtype=torch.long).to(device)
    d_ids = torch.tensor([corpus_ids[i] for i in candidate_ids],
                          dtype=torch.long).to(device)
    scores = cross_model(q_ids, d_ids).cpu().numpy()
    order  = np.argsort(-scores)
    return [candidate_ids[o] for o in order[:top_k]]


# Precision@5 before and after re-ranking
def precision_at_k(q_idx: int, retrieved: list, k: int = 5) -> float:
    relevant = {i for i, d in enumerate(domain_labels)
                if d == domain_labels[q_idx] and i != q_idx}
    return len(set(retrieved[:k]) & relevant) / k

prec_bm25, prec_reranked = [], []
for q_idx in range(N_DOCS):
    # Stage 1: BM25 top-20
    candidates = bm25_retrieve(corpus[q_idx], top_k=20)
    # Stage 2: cross-encoder re-rank
    reranked   = rerank_with_cross_encoder(corpus[q_idx], candidates, top_k=5)
    first_stage_top5 = candidates[:5]

    prec_bm25.append(precision_at_k(q_idx, first_stage_top5))
    prec_reranked.append(precision_at_k(q_idx, reranked))

print(f"\nPrecision@5 comparison:")
print(f"  BM25 first-stage  : {np.mean(prec_bm25):.3f}")
print(f"  After re-ranking  : {np.mean(prec_reranked):.3f}")

fig, ax = plt.subplots(figsize=(5, 3))
ax.bar(['BM25 (first stage)', 'Cross-encoder\n(re-ranked)'],
       [np.mean(prec_bm25), np.mean(prec_reranked)],
       color=['steelblue', 'seagreen'])
ax.set_ylabel('Precision@5')
ax.set_title('Re-ranking Effect on Precision@5')
ax.set_ylim(0, 1)
for i, v in enumerate([np.mean(prec_bm25), np.mean(prec_reranked)]):
    ax.text(i, v + 0.02, f'{v:.3f}', ha='center', fontsize=11)
plt.tight_layout()
plt.savefig('/tmp/08_rw2_reranking.png', dpi=100)
plt.close()
print("Plot saved to /tmp/08_rw2_reranking.png")

## Real-World Example 3: Evaluation Metrics from Scratch

Retrieval systems are evaluated with ranking-aware metrics that consider
where in the ranked list relevant documents appear:

- **MRR@K** (Mean Reciprocal Rank): average of 1/rank_first_relevant, capturing
  how high the first relevant document appears
- **NDCG@K** (Normalised Discounted Cumulative Gain): weights relevance by position
  using a logarithmic discount; robust to multiple relevant documents
- **Recall@K**: fraction of all relevant documents found in top K

## Comparison: BM25 vs Dense vs Hybrid

| Method | Strengths | Weaknesses |
|---|---|---|
| BM25 | Fast, exact keyword match, no training needed | Fails on synonyms, semantic gaps |
| Dense | Semantic understanding, handles paraphrases | Needs training data, slower indexing |
| Hybrid (RRF) | Best of both, robust to query type | Slightly higher latency, two systems |

In [ ]:
# Real-World Example 3: IR metrics from scratch + system comparison

def compute_mrr_at_k(retrieved: list, relevant: set, k: int = 10) -> float:
    """
    Mean Reciprocal Rank @K for a single query.
    Returns 1/rank of the first relevant document (0 if none in top K).
    """
    for rank, doc_id in enumerate(retrieved[:k], start=1):
        if doc_id in relevant:
            return 1.0 / rank
    return 0.0


def compute_dcg_at_k(retrieved: list, relevant: set, k: int = 10) -> float:
    """
    Discounted Cumulative Gain @K.
    Binary relevance: rel_i = 1 if doc in relevant else 0.
    DCG@K = sum_{i=1}^{K} rel_i / log2(i+1)
    """
    dcg = 0.0
    for rank, doc_id in enumerate(retrieved[:k], start=1):
        if doc_id in relevant:
            dcg += 1.0 / math.log2(rank + 1)
    return dcg


def compute_ndcg_at_k(retrieved: list, relevant: set, k: int = 10) -> float:
    """
    Normalised DCG@K: DCG / IDCG (ideal DCG assuming top-K are all relevant).
    """
    dcg  = compute_dcg_at_k(retrieved, relevant, k)
    # Ideal: all relevant docs appear first
    n_ideal = min(len(relevant), k)
    idcg = sum(1.0 / math.log2(i + 2) for i in range(n_ideal))
    return dcg / (idcg + 1e-8)


def compute_recall_at_k(retrieved: list, relevant: set, k: int = 20) -> float:
    """Recall@K: fraction of relevant documents found in the top K results."""
    if not relevant:
        return 1.0
    return len(set(retrieved[:k]) & relevant) / len(relevant)


def evaluate_system(retriever_fn, metric_k: int = 10) -> dict:
    """
    Evaluate a retrieval function over all queries.

    Args:
        retriever_fn: callable(query_words) -> ranked list of doc ids
        metric_k: cutoff for MRR and NDCG

    Returns:
        dict with MRR@K, NDCG@K, Recall@20
    """
    mrrs, ndcgs, rec20s = [], [], []
    for q_idx in range(N_DOCS):
        relevant = {i for i, d in enumerate(domain_labels)
                    if d == domain_labels[q_idx] and i != q_idx}
        if not relevant:
            continue
        ranked = retriever_fn(corpus[q_idx])
        mrrs.append(compute_mrr_at_k(ranked, relevant, k=metric_k))
        ndcgs.append(compute_ndcg_at_k(ranked, relevant, k=metric_k))
        rec20s.append(compute_recall_at_k(ranked, relevant, k=20))
    return {
        f'MRR@{metric_k}': float(np.mean(mrrs)),
        f'NDCG@{metric_k}': float(np.mean(ndcgs)),
        'Recall@20': float(np.mean(rec20s)),
    }

# Retriever wrappers
def bm25_retriever(query_words): return bm25_retrieve(query_words)
def dense_retriever(query_words): return dense_retrieve(query_words, top_k=N_DOCS)
def hybrid_retriever(query_words):
    bm25_r  = bm25_retrieve(query_words)
    dense_r = dense_retrieve(query_words, top_k=N_DOCS)
    return rrf_fusion([bm25_r, dense_r])

systems = {'BM25': bm25_retriever, 'Dense': dense_retriever, 'Hybrid': hybrid_retriever}
all_metrics = {}

print("Evaluation metrics (averaged over all queries):")
print(f"{'Method':<10}  {'MRR@10':>8}  {'NDCG@10':>8}  {'Recall@20':>10}")
print('-' * 45)
for name, fn in systems.items():
    m = evaluate_system(fn, metric_k=10)
    all_metrics[name] = m
    print(f"{name:<10}  {m['MRR@10']:>8.3f}  {m['NDCG@10']:>8.3f}  {m['Recall@20']:>10.3f}")

# Grouped bar chart comparison
metric_names = ['MRR@10', 'NDCG@10', 'Recall@20']
method_names = list(all_metrics.keys())
x = np.arange(len(metric_names))
width = 0.25
colors = ['steelblue', 'darkorange', 'seagreen']

fig, ax = plt.subplots(figsize=(8, 4))
for i, (method, col) in enumerate(zip(method_names, colors)):
    vals = [all_metrics[method][m] for m in metric_names]
    rects = ax.bar(x + i * width, vals, width, label=method, color=col)
    for rect in rects:
        ax.text(rect.get_x() + rect.get_width()/2,
                rect.get_height() + 0.01,
                f'{rect.get_height():.3f}', ha='center', fontsize=8)

ax.set_xticks(x + width)
ax.set_xticklabels(metric_names)
ax.set_ylabel('Score')
ax.set_title('BM25 vs Dense vs Hybrid: MRR@10, NDCG@10, Recall@20')
ax.set_ylim(0, 1.1)
ax.legend()
plt.tight_layout()
plt.savefig('/tmp/08_comparison.png', dpi=100)
plt.close()
print("\nComparison plot saved to /tmp/08_comparison.png")
print()
print("Key insight: Hybrid retrieval is robust across query types.")
print("Dense retrieval may underperform BM25 with minimal training data,")
print("but improves with more data and better encoders (e.g., BERT).")